<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/Capstone%20Projects/Smart%20Home%20Energy%20Usage%20Tracker/Week%203/Pyspark_aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyspark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving energy_logs_clean.csv to energy_logs_clean.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving devices.csv to devices.csv


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("SmartHomeEnergy").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.read.option("header", True).option("inferSchema", True)
      .csv("energy_logs_clean.csv"))
print("Rows:", df.count())
df.printSchema()
df.show(5)

Rows: 9853
root
 |-- log_id: integer (nullable = true)
 |-- device_id: string (nullable = true)
 |-- room_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- power_watts: double (nullable = true)

+------+---------+-------+-------------------+----------+-----------+
|log_id|device_id|room_id|          timestamp|energy_kwh|power_watts|
+------+---------+-------+-------------------+----------+-----------+
|     1|      D01|    R01|2026-09-01 00:00:00|    1.0275|     1027.5|
|     2|      D02|    R01|2026-09-01 00:00:00|    0.0024|        2.4|
|     3|      D03|    R01|2026-09-01 00:00:00|    0.0014|        1.4|
|     4|      D04|    R02|2026-09-01 00:00:00|    0.9964|      996.4|
|     5|      D05|    R02|2026-09-01 00:00:00|    4.0E-4|        0.4|
+------+---------+-------+-------------------+----------+-----------+
only showing top 5 rows


In [ ]:
df = (df.withColumn("hour", F.hour("timestamp"))
        .withColumn("period",
                    F.when((F.col("hour") >= 18) & (F.col("hour") <= 22), "peak")
                     .otherwise("off_peak")))
df.select("device_id", "timestamp", "hour", "period", "energy_kwh").show(8)

+---------+-------------------+----+--------+----------+
|device_id|          timestamp|hour|  period|energy_kwh|
+---------+-------------------+----+--------+----------+
|      D01|2026-09-01 00:00:00|   0|off_peak|    1.0275|
|      D02|2026-09-01 00:00:00|   0|off_peak|    0.0024|
|      D03|2026-09-01 00:00:00|   0|off_peak|    0.0014|
|      D04|2026-09-01 00:00:00|   0|off_peak|    0.9964|
|      D05|2026-09-01 00:00:00|   0|off_peak|    4.0E-4|
|      D06|2026-09-01 00:00:00|   0|off_peak|    0.0713|
|      D07|2026-09-01 00:00:00|   0|off_peak|    3.0E-4|
|      D08|2026-09-01 00:00:00|   0|off_peak|     0.071|
+---------+-------------------+----+--------+----------+
only showing top 8 rows


In [ ]:
devices = (spark.read.option("header", True).csv("devices.csv")
           .select("device_id", "device_name", "device_type"))

usage = (df.groupBy("device_id", "period")
           .agg(F.round(F.sum("energy_kwh"), 3).alias("kwh"))
           .groupBy("device_id")
           .pivot("period", ["peak", "off_peak"]).sum("kwh")
           .fillna(0)
           .withColumn("total_kwh", F.round(F.col("peak") + F.col("off_peak"), 3))
           .withColumn("peak_pct", F.round(100 * F.col("peak") / F.col("total_kwh"), 1))
           .join(devices, "device_id")
           .orderBy(F.desc("total_kwh")))
usage.show(20, truncate=False)

+---------+------+--------+---------+--------+---------------------+---------------+
|device_id|peak  |off_peak|total_kwh|peak_pct|device_name          |device_type    |
+---------+------+--------+---------+--------+---------------------+---------------+
|D01      |161.8 |109.44  |271.24   |59.7    |Living Room AC       |air_conditioner|
|D04      |31.467|201.208 |232.675  |13.5    |Bedroom 1 AC         |air_conditioner|
|D11      |5.88  |96.574  |102.454  |5.7     |Water Heater         |water_heater   |
|D08      |8.053 |31.443  |39.496   |20.4    |Refrigerator         |refrigerator   |
|D09      |10.427|17.722  |28.149   |37.0    |Microwave            |microwave      |
|D14      |1.49  |15.162  |16.652   |8.9     |Washing Machine      |washing_machine|
|D06      |3.751 |12.878  |16.629   |22.6    |Bedroom 2 Fan        |fan            |
|D02      |15.253|1.346   |16.599   |91.9    |Living Room TV       |television     |
|D12      |0.195 |14.366  |14.561   |1.3     |Office Laptop Charg

In [ ]:
top5 = usage.limit(5)
top5.select("device_id", "device_name", "total_kwh", "peak_pct").show(truncate=False)

usage.filter(F.col("peak_pct") > 50).select("device_id", "device_name", "peak_pct").show(truncate=False)

+---------+--------------+---------+--------+
|device_id|device_name   |total_kwh|peak_pct|
+---------+--------------+---------+--------+
|D01      |Living Room AC|271.24   |59.7    |
|D04      |Bedroom 1 AC  |232.675  |13.5    |
|D11      |Water Heater  |102.454  |5.7     |
|D08      |Refrigerator  |39.496   |20.4    |
|D09      |Microwave     |28.149   |37.0    |
+---------+--------------+---------+--------+

+---------+---------------+--------+
|device_id|device_name    |peak_pct|
+---------+---------------+--------+
|D01      |Living Room AC |59.7    |
|D02      |Living Room TV |91.9    |
|D10      |Kitchen Light  |54.5    |
|D05      |Bedroom 1 Light|90.8    |
|D07      |Bedroom 2 Light|86.0    |
+---------+---------------+--------+



In [ ]:
top5.coalesce(1).write.mode("overwrite").option("header", True).csv("top_devices_spark")
!ls top_devices_spark
!zip -r top_devices_spark.zip top_devices_spark
files.download("top_devices_spark.zip")
spark.stop()

part-00000-98af12ce-16d8-4604-a342-4e22bea798ea-c000.csv  _SUCCESS
  adding: top_devices_spark/ (stored 0%)
  adding: top_devices_spark/_SUCCESS (stored 0%)
  adding: top_devices_spark/.part-00000-98af12ce-16d8-4604-a342-4e22bea798ea-c000.csv.crc (stored 0%)
  adding: top_devices_spark/._SUCCESS.crc (stored 0%)
  adding: top_devices_spark/part-00000-98af12ce-16d8-4604-a342-4e22bea798ea-c000.csv (deflated 34%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>